# Dynamic World Asset Creator

Earth Engine uses a parallel processing system to carry out computation across a large number of machines. To enable such processing, Earth Engine takes advantage of standard techniques commonly used by functional languages, such as referential transparency and lazy evaluation, for significant optimization and efficiency gains. [Reference](https://developers.google.com/earth-engine/tutorials/tutorial_js_03)

This means our code will only be run once we produce a side effect, for instance printing a map. 

On the other hand, The Earth Engine platform has a number of quota limits in place to ensure that resources are distributed fairly across users. [Reference](https://developers.google.com/earth-engine/guides/usage) 

This quota limit can easily be reached when dealing with big datasets. The work around this limitation is to divide our workload into manageable batches and upload them individually, creating small assets with our intermediate computation. We do so by exporting this computations as tasks and saving the result as an asset in GEE Cloud. [Reference](https://developers.google.com/earth-engine/guides/processing_environments)



In [1]:
import ee
import os
from dotenv import load_dotenv
import numpy as np
load_dotenv()

# Initialize the Earth Engine module.
PROJECT_ID = os.getenv("PROJECT_ID")
ee.Initialize(project = os.getenv("PROJECT_ID"))

## Dynamic World Yearly Mode

One of the tasks that exceeded the quota limit was calculating the Yearly Mode of Dynamic World for Denmark.

This task as been performed for the years (Inventory):
* 2025
* 2024
* 2023
* 2022
* 2021

To perform it for another year make sure to change the `YEAR` value and update the inventory

In [2]:
YEAR = '21'
START = ee.Date('20'+ YEAR + '-01-01')
END = START.advance(1, 'year')
DENMARK = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Denmark')).geometry()

In [3]:
# define filter
col_filter = ee.Filter.And(
    ee.Filter.bounds(DENMARK), 
    ee.Filter.date(START, END),
)

# Get relevant DW images
dw_col = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(col_filter)

# Calculate Yearly mode
most_frequent_class = dw_col.select('label').reduce(ee.Reducer.mode())
most_frequent_class = most_frequent_class.rename(['label'])

The following code cell creates the exporting task for the image resulting from the aggregation of the yearly classifications. After Running it you should see a new task in the GEE Code Editor Task Manager.Please be mindful that we are talking about heavy computations here, calculating the mode of year in DW for Denmark takes over 30 min. We want to minimize the amount of tasks we perform. 

Leaving `task.start` line commented out to avoid starting unwanted tasks

In [ ]:
# Export Task
task = ee.batch.Export.image.toAsset(
    image = most_frequent_class,
    description = 'DW_Yearly_Mode_20' + YEAR,
    assetId = 'projects/' + PROJECT_ID + '/assets/dw_20' + YEAR + '_mode', 
    scale = 10,
    region = DENMARK,
    maxPixels = 1e13
)
# task.start()

Once the task is complete add the generated image to the ImageCollection `DenmarkYearMode`